# PyTorch张量基础操作验证
本文档为PyTorch张量核心操作的验证笔记，覆盖创建、运算、广播、格式转换、设备迁移等基础用法。官方API参考：https://pytorch-cn.readthedocs.io/zh/latest/package_references/torch-nn/

## 张量创建

In [8]:
import torch
print(f"PyTorch版本: {torch.__version__}")
print(f"CUDA是否可用: {torch.cuda.is_available()}")
print(f"CUDA版本: {torch.version.cuda}")
print(f"当前GPU型号: {torch.cuda.get_device_name(0)}")

x=torch.empty(5,3) #创建5x3未初始化tensor
print(x)

PyTorch版本: 2.11.0+cu130
CUDA是否可用: True
CUDA版本: 13.0
当前GPU型号: NVIDIA GeForce RTX 5060 Laptop GPU
tensor([[4.4771e-16, 7.2027e-43, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00]])


In [14]:
x=torch.rand(5,3)   #创建5x3随机tensor
x1=torch.randn(5,3)
print(x,'\n',x1)

tensor([[0.0791, 0.7056, 0.1043],
        [0.8526, 0.1013, 0.7227],
        [0.7600, 0.6870, 0.4808],
        [0.6327, 0.4962, 0.1469],
        [0.9188, 0.6976, 0.5187]]) 
 tensor([[-1.0270,  0.2549,  0.4872],
        [-1.0310, -0.0823, -1.2735],
        [ 0.5059, -0.4637, -1.7389],
        [-0.4588, -1.4768, -0.2319],
        [-0.3728, -0.0652,  0.0441]])


In [29]:
x=torch.zeros(5,3)  #全0
print(x)
x=torch.ones(5,3,dtype=int)  #全1
print(x)

tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([[1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1],
        [1, 1, 1]])


In [35]:
x=torch.tensor([5,3],dtype=torch.float64)  # 把 Python列表[5, 3] 转换成张量
print(x)

tensor([5., 3.], dtype=torch.float64)


In [42]:
x=torch.ones(5,3)
print(x.size())
print(x.shape)

torch.Size([5, 3])
torch.Size([5, 3])


## 张量基础运算

In [53]:
# 加法
x=torch.rand(5,3)
y=torch.rand(5,3)
print(x+y)  # 不修改原张量，返回一个全新的张量
print(torch.add(x,y)) # 和 x + y 完全等价
y.add_(x)   # 会覆盖 y 原来的数据
print(y)

tensor([[0.9697, 1.2890, 0.9497],
        [0.8704, 0.1511, 1.6283],
        [1.3544, 0.5888, 1.3449],
        [0.9899, 0.7832, 0.4063],
        [0.8188, 1.1848, 1.0959]])
tensor([[0.9697, 1.2890, 0.9497],
        [0.8704, 0.1511, 1.6283],
        [1.3544, 0.5888, 1.3449],
        [0.9899, 0.7832, 0.4063],
        [0.8188, 1.1848, 1.0959]])
tensor([[0.9697, 1.2890, 0.9497],
        [0.8704, 0.1511, 1.6283],
        [1.3544, 0.5888, 1.3449],
        [0.9899, 0.7832, 0.4063],
        [0.8188, 1.1848, 1.0959]])


In [55]:
# 索引
y=x[0,:]
print(x)
print(y)
y+=1
print(x)
print(y) # y 不是 x 的一份副本，它就是 x 第一行数据的 “别名” (改 y，就是在改 x 的第一行)

tensor([[1.0535, 1.9802, 1.3389],
        [0.7457, 0.0543, 0.8820],
        [0.7337, 0.4560, 0.7274],
        [0.6305, 0.4127, 0.1174],
        [0.3678, 0.3239, 0.4881]])
tensor([1.0535, 1.9802, 1.3389])
tensor([[2.0535, 2.9802, 2.3389],
        [0.7457, 0.0543, 0.8820],
        [0.7337, 0.4560, 0.7274],
        [0.6305, 0.4127, 0.1174],
        [0.3678, 0.3239, 0.4881]])
tensor([2.0535, 2.9802, 2.3389])


In [58]:
# 改变形状
y=x.view(15)
z=x.view(-1,5)  #-1对应的维度可由其余维度推算出来
# view 不会复制数据，它只是直接修改了 “读取内存的方式”. 优点：极快（瞬时完成），零显存开销
print(x.size(),y.size(),z.size())

torch.Size([5, 3]) torch.Size([15]) torch.Size([3, 5])


## 广播机制

In [63]:
# 形状不同的tensor之间进行运算
x=torch.arange(1,3).view(1,2)  # size(1,2)
print(x)
y=torch.arange(1,4).view(3,1)  # size(3,1)
print(y)

print(x+y)

tensor([[1, 2]])
tensor([[1],
        [2],
        [3]])
tensor([[2, 3],
        [3, 4],
        [4, 5]])


## 张量与NumPy格式互转

In [72]:
# numpy() 与 from_numpy() 实现张量与NumPy数组互转，二者共享底层内存，修改其中一个会同步影响另一个！

# tensor to numpy
a=torch.ones(5)
b=a.numpy()

a+=1
print(a)
print(b)


tensor([2., 2., 2., 2., 2.])
[2. 2. 2. 2. 2.]


In [73]:
# numpy to tensor
import numpy as np
a=np.ones(5)
b=torch.from_numpy(a) # 共享内存、零拷贝

a+=1
print(a)
print(b)

[2. 2. 2. 2. 2.]
tensor([2., 2., 2., 2., 2.], dtype=torch.float64)


In [74]:
# 对比：torch.tensor() 会拷贝数据，不共享内存
c=torch.tensor(a)  # torch.tensor() 会默认拷贝数据，不会共享内存
a+=1
print(a)
print(c)

[3. 3. 3. 3. 3.]
tensor([2., 2., 2., 2., 2.], dtype=torch.float64)


## CPU与GPU设备迁移

In [78]:
x=torch.ones(5)
if torch.cuda.is_available():
    device = torch.device("cuda") # GPU
    y = torch.ones_like(x, device=device) # 直接创建⼀个在GPU上的Tensor (形状同x的全1新张量)
    x = x.to(device) # 等价于 .to("cuda")
    z = x + y
    print(z)
    print(z.to("cpu", torch.double))
    # torch.float32 = 单精度（默认，简写 float）
    # torch.double  = 双精度（64 位，精度更高）

tensor([2., 2., 2., 2., 2.], device='cuda:0')
tensor([2., 2., 2., 2., 2.], dtype=torch.float64)
